# Jais Arabic Voice Chatbot — Google Colab, T4
Runs the full cascade pipeline locally on this VM's GPU (prepared for T4 / 16 GB):

`Browser (WebRTC) -> Jais-ASR (STT) -> Jais-3 (LLM) -> Audar-TTS-V1 (TTS)`

Run cells top-to-bottom (Run all). The last cell prints a Colab proxy URL - open it and speak Arabic.

> Runtime > Change runtime type > **GPU (T4)**

In [ ]:
#@title 1) GPU check & adaptive defaults {display-mode:"form"}
import subprocess, re

def gpu_state():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
             "--format=csv,noheader"]).decode()
        name, total, free = [x.strip() for x in out.split(",")]
        return name, int(re.sub(r"\D", "", total)) / 1024.0, int(re.sub(r"\D", "", free)) / 1024.0
    except Exception as e:
        print("No GPU detected:", e)
        return "", 0.0, 0.0

name, total_gb, free_gb = gpu_state()
print(f"GPU: {name} | total {total_gb:.1f} GB | free {free_gb:.1f} GB")
# T4 (capability 7.5) has no bf16 -> the app defaults to fp16 everywhere.
if free_gb and free_gb < 20:
    print("Less than 20 GB free -> JAIS_ASR_MODEL=auto selects the W4A16-AWQ checkpoint.")
else:
    print("20+ GB free -> JAIS_ASR_MODEL=auto selects the fp16 Jais-ASR checkpoint.")

In [ ]:
#@title 2) Clone repo & install dependencies (~5-8 min) {display-mode:"form"}
REPO_URL = "https://github.com/USERNAME/jais-voice-chatbot.git" #@param {type:"string"}

import os, subprocess, sys
repo_dir = "/content/jais-voice-chatbot"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
os.chdir(repo_dir)
print("cwd:", os.getcwd())

# Runner extra pulls the prebuilt WebRTC UI; compressed-tensors is required by
# the W4A16-AWQ ASR checkpoint auto-selected on 16GB GPUs.
!pip install -q -r requirements.txt
print("Install done.")

In [ ]:
#@title 3) (Optional) Hugging Face token {display-mode:"form"}
import os
TOKEN = "" #@param {type:"string"}
# Not strictly required: all three repos are public. Set only if rate-limited.
if TOKEN:
    os.environ["HF_TOKEN"] = TOKEN
    from huggingface_hub import login
    login(TOKEN, add_to_git_credential=False)
else:
    print("No token set (fine for public repos).")

In [ ]:
#@title 4) Start the voice server {display-mode:"form"}
APP_PORT = 8080 #@param {type:"integer"}

import os, subprocess
os.environ.setdefault("JAIS_ASR_MODEL", "auto")   # auto -> AWQ on T4 (~3.4 GB)
os.environ.setdefault("JAIS_LLM_DTYPE", "fp16")
os.environ.setdefault("JAIS_TTS_VOICE", "demo_female_1")

logf = open("/content/jais-voice.log", "a")
proc = subprocess.Popen(
    ["python", "app.py", "-t", "webrtc", "--host", "0.0.0.0", "--port", str(APP_PORT)],
    stdout=logf, stderr=logf,
)
print("server started (PID %d); model load happens on first connection" % proc.pid)

In [ ]:
#@title 5) Expose the app & get your link {display-mode:"form"}
#@markdown Uses Colab's own proxy - no accounts or tokens needed.
import time, requests
from google.colab.output import serve_kernel_port_as_forward

for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{APP_PORT}/docs", timeout=2).status_code == 200:
            break
    except Exception:
        time.sleep(2)
else:
    print("Uvicorn not up yet - check /content/jais-voice.log, then re-run this cell.")

print(f"\n=== Click 'open /client' below or serve_kernel_port_as_window for pop-out ===")
serve_kernel_port_as_forward(port=APP_PORT, path="/client")

## Troubleshooting

- **OOM**: keep `JAIS_ASR_MODEL=auto` (AWQ ~3.4GB); lower `JAIS_LLM_MAX_NEW_TOKENS`.
- **Slow replies on T4**: expected with the quantized 6B ASR (3-8 s per turn).
- **`TTS emitted no speech tokens`**: reference bootstrap failed - set `JAIS_TTS_REF_WAV` + `JAIS_TTS_REF_TEXT` in cell 4.
- Browser cannot connect: use the printed proxy URL twice (first click warms models).
